# ETL Superstore → Star Schema

Sebelum mulai, jalankan di pgAdmin: `sql/01_create_database.sql`, lalu `sql/02_create_tables.sql` (di database `dwh`).

In [ ]:
import os, sys

# Java dari env conda (supaya terbaca juga saat dijalankan di VS Code)
os.environ.setdefault("JAVA_HOME", os.path.join(sys.prefix, "Library" if os.name == "nt" else "lib/jvm"))

In [ ]:
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

load_dotenv('../.env')

spark = SparkSession.builder \
    .appName("ETLSuperstore") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.14") \
    .getOrCreate()

# Extract

In [ ]:
def read_csv(file_name):
    return spark.read.csv(f'../data/{file_name}', header=True, inferSchema=True, escape='"')

def to_snake_case(df):
    new_names = [c.lower().replace(' ', '_').replace('-', '_').replace('/', '_') for c in df.columns]
    return df.toDF(*new_names)

orders = to_snake_case(read_csv('superstore_orders.csv'))
customers = to_snake_case(read_csv('superstore_customers.csv'))
products = to_snake_case(read_csv('superstore_products.csv'))

orders.count(), customers.count(), products.count()

# Transform

## Cek Data Kotor

In [ ]:
customers.groupby('segment').count().show()
orders[['region']].distinct().show()

## Bersihkan

In [ ]:
customers_clean = (customers
    .drop_duplicates()
    .withColumn('customer_name', F.trim('customer_name'))
    .withColumn('segment', F.initcap(F.trim('segment')))
    .drop_duplicates(subset=['customer_id']))

customers_clean.groupby('segment').count().show()

In [ ]:
products_clean = (products
    .drop_duplicates()
    .withColumn('category', F.initcap(F.trim('category')))
    .drop_duplicates(subset=['product_id']))

products_clean.groupby('category').count().show()

In [ ]:
orders_clean = (orders
    .drop_duplicates()
    .withColumn('region', F.initcap(F.trim('region')))
    .withColumn('postal_code', F.lpad(F.col('postal_code').astype('string'), 5, '0'))
    .fillna({'postal_code': 'Unknown'})
    .withColumn('order_date', F.to_date('order_date', 'M/d/yyyy'))
    .withColumn('ship_date', F.to_date('ship_date', 'M/d/yyyy'))
    .dropna(subset=['sales']))

orders.count(), orders_clean.count()

## Buang Order yang Produknya Tidak Ada

In [ ]:
orders_clean.join(products_clean, on='product_id', how='left_anti').show()

In [ ]:
orders_valid = orders_clean.join(products_clean[['product_id']], on='product_id', how='inner')
orders_clean.count(), orders_valid.count()

## Dimension

In [ ]:
dim_customer = customers_clean[['customer_id', 'customer_name', 'segment']]
dim_product = products_clean[['product_id', 'product_name', 'category', 'sub_category']]

dim_customer.show(5)
dim_product.show(5)

In [ ]:
location_cols = ['country_region', 'region', 'state', 'city', 'postal_code']

dim_location = orders_valid[location_cols].drop_duplicates() \
    .withColumn('location_id', F.row_number().over(Window.orderBy('state', 'city', 'postal_code')))
dim_location.show(5)

In [ ]:
dates = orders_valid.select(F.col('order_date').alias('full_date')) \
    .union(orders_valid.select(F.col('ship_date').alias('full_date'))) \
    .distinct()

dim_date = dates \
    .withColumn('date_id', F.date_format('full_date', 'yyyyMMdd').cast('int')) \
    .withColumn('year', F.year('full_date')) \
    .withColumn('quarter', F.quarter('full_date')) \
    .withColumn('month', F.month('full_date')) \
    .withColumn('month_name', F.date_format('full_date', 'MMMM')) \
    .withColumn('day_name', F.date_format('full_date', 'EEEE'))
dim_date.show(5)

## Fact

In [ ]:
fact_sales = orders_valid \
    .join(dim_location, on=location_cols, how='left') \
    .withColumn('order_date_id', F.date_format('order_date', 'yyyyMMdd').cast('int')) \
    .withColumn('ship_date_id', F.date_format('ship_date', 'yyyyMMdd').cast('int')) \
    .select('row_id', 'order_id', 'order_date_id', 'ship_date_id', 'customer_id', 'product_id',
            'location_id', 'ship_mode', 'sales', 'quantity', 'discount', 'profit')
fact_sales.show(5)

In [ ]:
fact_sales.count()

# Load to Postgre

In [ ]:
# PostgreSQL JDBC Connection (isi ada di file .env)
postgres_url = f"jdbc:postgresql://{os.getenv('PG_HOST')}:{os.getenv('PG_PORT')}/{os.getenv('PG_DB')}"
postgres_properties = {
    "user": os.getenv("PG_USER"),
    "password": os.getenv("PG_PASSWORD"),
    "driver": "org.postgresql.Driver"
}

In [ ]:
# dimension dulu, baru fact
dim_customer.write.jdbc(url=postgres_url, table="dim_customer", mode="append", properties=postgres_properties)
dim_product.write.jdbc(url=postgres_url, table="dim_product", mode="append", properties=postgres_properties)
dim_location.write.jdbc(url=postgres_url, table="dim_location", mode="append", properties=postgres_properties)
dim_date.write.jdbc(url=postgres_url, table="dim_date", mode="append", properties=postgres_properties)
fact_sales.write.jdbc(url=postgres_url, table="fact_sales", mode="append", properties=postgres_properties)

# Load Data from Postgre

In [ ]:
sql_query = """SELECT
    l.region,
    p.category,
    ROUND(SUM(f.sales), 2)::float AS total_sales
FROM fact_sales f
JOIN dim_location l ON f.location_id = l.location_id
JOIN dim_product p ON f.product_id = p.product_id
GROUP BY 1, 2
ORDER BY 1, 2"""
query = f"({sql_query}) AS subquery"

df = spark.read.jdbc(url=postgres_url, table=query, properties=postgres_properties)
df.show()

In [ ]:
spark.stop()